# **RAGdemo**

Implements a retrieval-augmented generation workflow combining sentence embeddings, FAISS vector search, and the chat model.

Pipeline overview:
1. **Embedding model** – `create_embedding_model()` loads `all-MiniLM-L6-v2` from Sentence Transformers and prepares it for inference.
2. **Document dataclasses** – `DocChunk` stores each chunk’s text, embedding, and identifiers while `lookupQuery` wraps incoming user questions.
3. **Chunking and indexing** – `preprocess_pages2chunks()` trims input records, `load_doc_from_path()` ingests a JSONL file, creates dense embeddings, and builds a FAISS inner-product index (with L2 normalization for cosine similarity).
4. **Retrieval** – `retrieve_relevant_docs()` searches the index for the top-`k` passages, deduplicates them, and returns rich chunk objects.
5. **Prompt construction** – Retrieved passages are concatenated into a context block that is injected into a prompt template instructing the model to cite references explicitly.
6. **Generation** – `rag_ask()` orchestrates retrieval + generation and returns both the response and the supporting documents.
7. **Demo run** – Sample questions about National Cheng Kung University show how the model uses the retrieved knowledge to answer factual questions more reliably than pure inference.

## Environment Setup

In [ ]:
!pip install faiss-cpu==1.11.0.post1

!mkdir -p demo_dataset && cd demo_dataset && \
wget -nc https://raw.githubusercontent.com/yasaisen/LLMTutorial/main/demo_dataset/ncku_wikipedia_2510080406.jsonl # 成大wikipedia paragraph 標頭是text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 61.4 MB/s eta 0:00:00
--2025-11-24 21:15:16--  https://raw.githubusercontent.com/yasaisen/LLMTutorial/main/demo_dataset/ncku_wikipedia_2510080406.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8767 (8.6K) [text/plain]
Saving to: ‘ncku_wikipedia_2510080406.jsonl’

ncku_wikipedia_2510 100%[===================>]   8.56K  --.-KB/s    in 0s      

2025-11-24 21:15:16 (83.7 MB/s) - ‘ncku_wikipedia_2510080406.jsonl’ saved [8767/8767]



In [ ]:
# please enter your Hugging Face token locally before running
!huggingface-cli login --token hf_xxxxxxxxxxxxxxxxx

## General Methods Define

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
def create_model(
    lm_model_name = "google/gemma-3-4b-it",
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
):
    tokenizer = AutoTokenizer.from_pretrained(lm_model_name)
    model = AutoModelForCausalLM.from_pretrained(
        lm_model_name,
        device_map="auto",
    ).eval()

    return model, tokenizer

In [ ]:
def lm_template(
    text: str,
    system_prompt: str = "You are a helpful assistant.",
):
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_prompt}]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": text}]
        }
    ]

In [ ]:
@torch.inference_mode()
def generate(
    prompt,
    tokenizer,
    model,
    max_new_tokens: int = 256,
    temperature: float = 1,
):
    inputs = tokenizer.apply_chat_template(
        prompt,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {
        k: (
            v.to(model.device, dtype=model.dtype)
            if v.dtype.is_floating_point else v.to(model.device)
        )
        for k, v in inputs.items()
    }

    input_len = inputs["input_ids"].shape[-1]

    # max_len = int(model.config.text_config.max_position_embeddings)
    max_len = int(model.config.max_position_embeddings) # use 1b for colab..
    if input_len > max_len:
        raise ValueError(
            f"Input length {input_len} exceeds maximum allowed length of {max_len} tokens."
        )

    generation = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
    )
    generation = generation[0][input_len:]

    response = tokenizer.decode(
        generation,
        skip_special_tokens=True
    )

    return response

## RAG Methods Define

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from dataclasses import dataclass
import json
from typing import List

In [ ]:
def create_embedding_model(
    emb_model_name: str = "all-MiniLM-L6-v2",
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
):
    embedding_model = SentenceTransformer(
        emb_model_name
    ).to(device).eval()

    return embedding_model

In [ ]:
def query_template(
    query,
):
    return f"""
{query.question}
"""

def prompt_template(
    context,
    query,
):
    return f"""
References:
{context}
Question:
{query.question}
Do not use markdown syntax to answer and put the answer after "Answer:"
"""

@dataclass
class DocChunk:
    idx: int
    content: str
    embedding: np.ndarray = None
    score = None

@dataclass
class lookupQuery:
    question: str

In [ ]:
def preprocess_pages2chunks(
    pages_list,
):
    chunked_list = []
    for page in pages_list:

        chunked_list.append({
            'text': page['text'].strip(),
        })

    idx = 0
    all_chunks = []
    for chunked in chunked_list:
        doc = DocChunk(
            idx=idx,
            content=chunked['text'],
        )
        all_chunks += [doc]
        idx += 1

    return all_chunks

def load_doc_from_path(
    documents_path: str,
    embedding_model,
):
    pages_list = []
    with open(documents_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if line:
                data = json.loads(line)
                pages_list.append(data)

    chunk_list = preprocess_pages2chunks(
        pages_list=pages_list
    )
    contents = [doc.content for doc in chunk_list]

    embeddings = embedding_model.encode(contents)
    index = faiss.IndexFlatIP(embeddings.shape[1]) # Create FAISS index
    faiss.normalize_L2(embeddings)
    index.add(embeddings.astype(np.float32))

    # Save documents
    for doc, embedding in zip(chunk_list, embeddings):
        doc.embedding = embedding

    return chunk_list, index

def retrieve_relevant_docs(
    search_query: str,
    embedding_model,
    index,
    chunk_list,
    top_k: int = 3
) -> List[DocChunk]:

    relevant_docs = []
    query_embedding = embedding_model.encode([search_query])
    faiss.normalize_L2(query_embedding)
    scores, indices = index.search(query_embedding.astype(np.float32), top_k)

    for score, idx in zip(scores[0], indices[0]):
        if idx < len(chunk_list):
            doc = chunk_list[idx]
            relevant_docs.append(doc)

    seen = set()
    unique_data = []
    for doc in relevant_docs:
        if doc.idx not in seen:
            seen.add(doc.idx)
            unique_data.append(doc)

    return unique_data

In [ ]:
def rag_ask(
    user_query: str,
    model,
    tokenizer,
    embedding_model,
    index,
    chunk_list,
    top_k: int = 3,
    max_new_tokens: int = 256,
    temperature: float = 1,
) -> str:
    query = lookupQuery(
        question=user_query,
    )
    search_query = query_template(
        query=query,
    )
    relevant_docs = retrieve_relevant_docs(
        search_query=search_query,
        embedding_model=embedding_model,
        index=index,
        chunk_list=chunk_list,
        top_k=top_k,
    )

    context = ""
    for i, doc in enumerate(relevant_docs):
        context += f"References {i+1}:{doc.content}\n"

    text = prompt_template(
        context=context,
        query=query,
    )
    prompt = lm_template(
        text=text
    )
    response = generate(
        prompt=prompt,
        tokenizer=tokenizer,
        model=model,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
    )

    return {
        'response': response,
        'prompt': prompt,
        'relevant_docs': relevant_docs,
    }

## Create Models & load documents

In [ ]:
model, tokenizer = create_model(
    lm_model_name="google/gemma-3-1b-it" # use 1b for colab..
)
embedding_model = create_embedding_model(
    emb_model_name="all-MiniLM-L6-v2"
)

documents_path = './demo_dataset/ncku_wikipedia_2510080406.jsonl'

chunk_list, index = load_doc_from_path(
    documents_path=documents_path,
    embedding_model=embedding_model,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Cases Test

In [ ]:
test_cases = [
    "When was National Cheng Kung University established?",
    "Who is the current president of National Cheng Kung University?",
    "Where is National Cheng Kung University located?",
    "What is the English translation of NCKU's motto?",
]

In [ ]:
print("Start Demo!\n")

for i, query in enumerate(test_cases, 1):
    print(f"Test Case ({i}) {'=' * 50}")
    print(f"user input: {query}")

    response = rag_ask(
        user_query=query,
        model=model,
        tokenizer=tokenizer,
        embedding_model=embedding_model,
        index=index,
        chunk_list=chunk_list,
    )['response']
    print(f"Model response: {response}\n{'=' * 64}\n")

print("\nDemo Completed!")

Start Demo!

Test Case (1) ==================================================
user input: When was National Cheng Kung University established?
Model response: The National Cheng Kung University was established in January 1931 as the Tainan Technical College. Answer: January 1931

Test Case (2) ==================================================
user input: Who is the current president of National Cheng Kung University?
Model response: Former Minister for Education Wu Jin served as the first president of the new National Cheng Kung University. Answer: Wu Jin

Test Case (3) ==================================================
user input: Where is National Cheng Kung University located?
Model response: Answer: The National Cheng Kung University is located in the East District of Tainan, Taiwan.

Test Case (4) ==================================================
user input: What is the English translation of NCKU's motto?
Model response: Answer: The English translation of NCKU’s motto is not ex

In [ ]:
# === Step 1. QA dataset ===

dataset = [
    {
        "question": "Which international universities have signed dual degree or collaboration agreements with NCKU?",
        "answer": "KU Leuven and IMEC signed a dual degree agreement with NCKU in 2018, and Purdue University also signed a collaboration agreement for international dual degree programs and online courses."
    },
    {
        "question": "What special research program does NCKU participate in that is related to Academia Sinica?",
        "answer": "NCKU participates in the Taiwan International Graduate Program in Interdisciplinary Neuroscience of Academia Sinica."
    },
    {
        "question": "How is NCKU ranked globally according to the QS World University Rankings 2024?",
        "answer": "NCKU is ranked 228th in the QS World University Rankings 2024."
    },
    {
        "question": "What is one engineering-related achievement or ranking that NCKU holds according to ARWU or ESI?",
        "answer": "The Academic Ranking of World Universities ranked NCKU 76–100th in engineering in 2016, and the ESI database ranked NCKU 24th in engineering based on publication impact."
    },
    {
        "question": "Which notable architect associated with NCKU designed Taipei 101?",
        "answer": "Architect Chu-yuan Lee, who studied at NCKU, designed Taipei 101."
    },
    {
        "question": "What is the main language of instruction at NCKU, and are there courses offered in other languages?",
        "answer": "Most courses at NCKU are taught in Mandarin Chinese, but many are also offered in English."
    },
    {
        "question": "What makes the Y. S. Sun Green Building Research Center notable in terms of sustainability?",
        "answer": "It is the world's first green educational center and Taiwan's first zero-carbon building."
    },
    {
        "question": "Which Taiwanese political figure graduated from NCKU’s post-baccalaureate medicine program?",
        "answer": "Lai Ching-te, the president of Taiwan, graduated from NCKU’s post-baccalaureate medicine program in 1991."
    }
]

questions = [x["question"] for x in dataset]
gold_answers = [x["answer"] for x in dataset]

len(dataset), questions[0], gold_answers[0]


(8,
 'Which international universities have signed dual degree or collaboration agreements with NCKU?',
 'KU Leuven and IMEC signed a dual degree agreement with NCKU in 2018, and Purdue University also signed a collaboration agreement for international dual degree programs and online courses.')

In [ ]:
# === Step 2. Non-RAG baseline（No RAG） ===

def generate_answer_non_rag(question: str, max_new_tokens: int = 128) -> str:

    prompt = (
        "Answer the following question about National Cheng Kung University (NCKU) "
        "as accurately as possible.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 避免把整個 prompt 都印出來，切掉前面的部分
    if "Answer:" in text:
        text = text.split("Answer:", 1)[-1]
    return text.strip()

pred_answers_non_rag = []
for q in questions:
    ans = generate_answer_non_rag(q)
    pred_answers_non_rag.append(ans)
    print("Q:", q)
    print("Pred (non-RAG):", ans)
    print("-" * 80)


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: Which international universities have signed dual degree or collaboration agreements with NCKU?
Pred (non-RAG): The following international universities have signed dual degree or collaboration agreements with NCKU:

*   **University of California, Berkeley**
*   **University of Oxford**
*   **University of Tokyo**
*   **University of Michigan**
*   **University of Toronto**
*   **University of Melbourne**
*   **University of Sydney**
*   **University of Hong Kong**
*   **University of Zurich**
*   **University of Vienna**
*   **University of Salamanca**
*   **University of Rome**
*   **University of Madrid**
*   **
--------------------------------------------------------------------------------
Q: What special research program does NCKU participate in that is related to Academia Sinica?
Pred (non-RAG): NCKU's research program is the "Research on the History and Culture of the Chinese Mountain and the Development of the Chinese Mountain."
**Explanation:**
NCKU's research program is 

In [ ]:
# === Step 3. RAG pipeline Answer ===

pred_answers_rag = []
for q in questions:
    ans = rag_ask(
        user_query=q,
        model=model,
        tokenizer=tokenizer,
        embedding_model=embedding_model,
        index=index,
        chunk_list=chunk_list,
    )['response'] # Retrieve only the response part
    pred_answers_rag.append(ans)
    print("Q:", q)
    print("Pred (RAG):", ans)
    print("-" * 80)

Q: Which international universities have signed dual degree or collaboration agreements with NCKU?
Pred (RAG): Answer: According to the provided references, NCKU has signed dual degree or collaboration agreements with the following international universities: University of Southern California, Technical University of Munich, Leiden University, the University of New South Wales, Kyoto University, Seoul National University, National University of Singapore, and IMEC.

--------------------------------------------------------------------------------
Q: What special research program does NCKU participate in that is related to Academia Sinica?
Pred (RAG): NCKU participates in the Taiwan International Graduate Program in Interdisciplinary Neuroscience of Academia Sinica.
Answer: NCKU participates in the Taiwan International Graduate Program in Interdisciplinary Neuroscience of Academia Sinica.
--------------------------------------------------------------------------------
Q: How is NCKU rank

In [ ]:
# === Step 4. Non-RAG vs RAG：BLEU / ROUGE / BERTScore ===

!pip install -q evaluate rouge-score bert-score

import evaluate
import numpy as np

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# BLEU
bleu_non_rag = bleu.compute(
    predictions=pred_answers_non_rag,
    references=[[g] for g in gold_answers]
)
bleu_rag = bleu.compute(
    predictions=pred_answers_rag,
    references=[[g] for g in gold_answers]
)

# ROUGE (看 ROUGE-L)
rouge_non_rag = rouge.compute(
    predictions=pred_answers_non_rag,
    references=gold_answers
)
rouge_rag = rouge.compute(
    predictions=pred_answers_rag,
    references=gold_answers
)

# BERTScore（使用 roberta-large）
bertscore_non_rag = bertscore.compute(
    predictions=pred_answers_non_rag,
    references=gold_answers,
    lang="en"
)
bertscore_rag = bertscore.compute(
    predictions=pred_answers_rag,
    references=gold_answers,
    lang="en"
)

print("=== BLEU ===")
print("Non-RAG:", bleu_non_rag)
print("RAG    :", bleu_rag)

print("\n=== ROUGE ===")
print("Non-RAG:", rouge_non_rag)
print("RAG    :", rouge_rag)

print("\n=== BERTScore (F1 mean) ===")
print("Non-RAG F1 mean:",
      float(np.mean(bertscore_non_rag["f1"])))
print("RAG F1 mean    :",
      float(np.mean(bertscore_rag["f1"])))


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.4 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== BLEU ===
Non-RAG: {'bleu': 0.017931805954989506, 'precisions': [0.0774487471526196, 0.021839080459770115, 0.010440835266821345, 0.00585480093676815], 'brevity_penalty': 1.0, 'length_ratio': 5.9324324324324325, 'translation_length': 878, 'reference_length': 148}
RAG    : {'bleu': 0.13193603839464693, 'precisions': [0.30578512396694213, 0.1452991452991453, 0.09292035398230089, 0.07339449541284404], 'brevity_penalty': 1.0, 'length_ratio': 1.635135135135135, 'translation_length': 242, 'reference_length': 148}

=== ROUGE ===
Non-RAG: {'rouge1': np.float64(0.17081411722829756), 'rouge2': np.float64(0.04795688136727945), 'rougeL': np.float64(0.13989192240400491), 'rougeLsum': np.float64(0.15002697233790868)}
RAG    : {'rouge1': np.float64(0.4650441510753629), 'rouge2': np.float64(0.27273270250645093), 'rougeL': np.float64(0.4209917920656635), 'rougeLsum': np.float64(0.4211152367923914)}

=== BERTScore (F1 mean) ===
Non-RAG F1 mean: 0.8342724516987801
RAG F1 mean    : 0.8960192203521729


# Qwen-0.5B-Instruct

In [19]:
!pip install -q transformers accelerate sentencepiece

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME_QWEN = "Qwen/Qwen2-0.5B-Instruct"  # 模型名稱

tokenizer_qwen = AutoTokenizer.from_pretrained(MODEL_NAME_QWEN)
model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_QWEN,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [21]:
# Non-RAG
def generate_answer_non_rag_qwen(question: str, max_new_tokens: int = 128) -> str:
    prompt = (
        "You are a helpful assistant answering questions about National Cheng Kung University (NCKU).\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer_qwen(prompt, return_tensors="pt").to(model_qwen.device)
    with torch.no_grad():
        outputs = model_qwen.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    text = tokenizer_qwen.decode(outputs[0], skip_special_tokens=True)
    if "Answer:" in text:
        text = text.split("Answer:", 1)[-1]
    return text.strip()


In [22]:
pred_answers_non_rag_qwen = []
for q in questions:
    ans = generate_answer_non_rag_qwen(q)
    pred_answers_non_rag_qwen.append(ans)
    print("Q:", q)
    print("Pred (Qwen non-RAG):", ans)
    print("-" * 80)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: Which international universities have signed dual degree or collaboration agreements with NCKU?
Pred (Qwen non-RAG): The Chinese Academy of Sciences, the European Union, and the United States Department of State.
Is this answer correct? Yes or No. Yes.
--------------------------------------------------------------------------------
Q: What special research program does NCKU participate in that is related to Academia Sinica?
Pred (Qwen non-RAG): The Special Research Program of the Institute of Advanced Study at NCKU is related to Academia Sinica. It is an interdisciplinary program that combines academic and scientific research with social, cultural, political, and economic activities.

Is this answer correct? Yes.
This answer is correct. The Special Research Program of the Institute of Advanced Study at NCKU is indeed related to Academia Sinica, which is one of the main objectives of the university. This program aims to promote cross-disciplinary collaboration between academia and in

In [23]:
# RAG
def generate_answer_rag_qwen(question: str, top_k: int = 3, max_new_tokens: int = 128) -> str:
    # First, perform retrieval using the existing functions and models
    query_obj = lookupQuery(question=question) # Use the existing lookupQuery dataclass
    search_query = query_template(query=query_obj) # Use the existing query_template

    # Call the correct retrieval function
    relevant_docs = retrieve_relevant_docs(
        search_query=search_query,
        embedding_model=embedding_model, # Use the globally available embedding_model
        index=index, # Use the globally available index
        chunk_list=chunk_list, # Use the globally available chunk_list
        top_k=top_k,
    )

    context = ""
    for i, doc in enumerate(relevant_docs):
        context += f"References {i+1}:{doc.content}\n"

    # Then, construct the prompt for Qwen
    prompt_text = (
        "You are a helpful assistant answering questions about National Cheng Kung University (NCKU).\n"
        "Use ONLY the following context to answer the question. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    inputs = tokenizer_qwen(prompt_text, return_tensors="pt").to(model_qwen.device)
    with torch.no_grad():
        outputs = model_qwen.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    text = tokenizer_qwen.decode(outputs[0], skip_special_tokens=True)
    if "Answer:" in text:
        text = text.split("Answer:", 1)[-1]
    return text.strip()


In [24]:
pred_answers_rag_qwen = []
for q in questions:
    ans = generate_answer_rag_qwen(q, top_k=3)
    pred_answers_rag_qwen.append(ans)
    print("Q:", q)
    print("Pred (Qwen RAG):", ans)
    print("-" * 80)


Q: Which international universities have signed dual degree or collaboration agreements with NCKU?
Pred (Qwen RAG): University of Southern California, Technical University of Munich, Leiden University, the University of New South Wales, Kyoto University, Seoul National University, National University of Singapore, etc.
--------------------------------------------------------------------------------
Q: What special research program does NCKU participate in that is related to Academia Sinica?
Pred (Qwen RAG): NCKU participates in the Taiwan International Graduate Program in Interdisciplinary Neuroscience of Academia Sinica, the national academy of Taiwan.
The answer is not explicitly stated in the context but it can be inferred based on the information provided. NCKU is part of the Taiwan International Graduate Program in Interdisciplinary Neuroscience of Academia Sinica, which is similar to the Top Global University Project in Japan and Universities of Excellence in Germany. This sugges

In [25]:
# BLEU / ROUGE / BERTScore
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [26]:
# Qwen Non-RAG
bleu_non_rag_qwen = bleu.compute(
    predictions=pred_answers_non_rag_qwen,
    references=[[g] for g in gold_answers]
)
rouge_non_rag_qwen = rouge.compute(
    predictions=pred_answers_non_rag_qwen,
    references=gold_answers
)
bertscore_non_rag_qwen = bertscore.compute(
    predictions=pred_answers_non_rag_qwen,
    references=gold_answers,
    lang="en"
)

# Qwen RAG
bleu_rag_qwen = bleu.compute(
    predictions=pred_answers_rag_qwen,
    references=[[g] for g in gold_answers]
)
rouge_rag_qwen = rouge.compute(
    predictions=pred_answers_rag_qwen,
    references=gold_answers
)
bertscore_rag_qwen = bertscore.compute(
    predictions=pred_answers_rag_qwen,
    references=gold_answers,
    lang="en"
)

print("=== Qwen Non-RAG ===")
print("BLEU:", bleu_non_rag_qwen)
print("ROUGE:", rouge_non_rag_qwen)
print("BERTScore F1 mean:",
      sum(bertscore_non_rag_qwen["f1"]) / len(bertscore_non_rag_qwen["f1"]))

print("\n=== Qwen RAG ===")
print("BLEU:", bleu_rag_qwen)
print("ROUGE:", rouge_rag_qwen)
print("BERTScore F1 mean:",
      sum(bertscore_rag_qwen["f1"]) / len(bertscore_rag_qwen["f1"]))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== Qwen Non-RAG ===
BLEU: {'bleu': 0.028467892389144538, 'precisions': [0.11754684838160136, 0.044905008635578586, 0.017513134851138354, 0.007104795737122558], 'brevity_penalty': 1.0, 'length_ratio': 3.9662162162162162, 'translation_length': 587, 'reference_length': 148}
ROUGE: {'rouge1': np.float64(0.24450295078053588), 'rouge2': np.float64(0.11673665534763823), 'rougeL': np.float64(0.21025962016199018), 'rougeLsum': np.float64(0.22518478601748076)}
BERTScore F1 mean: 0.8616043999791145

=== Qwen RAG ===
BLEU: {'bleu': 0.06387035110970873, 'precisions': [0.150093808630394, 0.07047619047619047, 0.04448742746615087, 0.03536345776031434], 'brevity_penalty': 1.0, 'length_ratio': 3.6013513513513513, 'translation_length': 533, 'reference_length': 148}
ROUGE: {'rouge1': np.float64(0.22607498533010043), 'rouge2': np.float64(0.11968774086243968), 'rougeL': np.float64(0.19873838863608684), 'rougeLsum': np.float64(0.19587720463679542)}
BERTScore F1 mean: 0.8634777963161469


# Custom RAG Pipeline

## 改變 Pipeline 1 : top_k 從 3 改成 5

In [27]:
# top_k 從 3 改成 5
pred_answers_rag_top5 = []
for q in questions:
    ans = rag_ask(
        user_query=q,
        model=model,
        tokenizer=tokenizer,
        embedding_model=embedding_model,
        index=index,
        chunk_list=chunk_list,
        top_k=5
    )['response']
    pred_answers_rag_top5.append(ans)
    print("Q:", q)
    print("Pred (RAG, top_k=5):", ans)
    print("-" * 80)


Q: Which international universities have signed dual degree or collaboration agreements with NCKU?
Pred (RAG, top_k=5): References 1: University of Southern California, Technical University of Munich, Leiden University, the University of New South Wales, Kyoto University, Seoul National University, National University of Singapore, etc.
References 2: KU Leuven, IMEC
References 3: TU Darmstadt
References 4: National Taiwan University, National Tsing Hua University, and National Yang Ming Chiao Tung University
References 5: National Taiwan University, National Tsing Hua University, and National Yang Ming Chiao Tung University

--------------------------------------------------------------------------------
Q: What special research program does NCKU participate in that is related to Academia Sinica?
Pred (RAG, top_k=5): NCKU participates in the Taiwan International Graduate Program in Interdisciplinary Neuroscience of Academia Sinica.
Answer: NCKU participates in the Taiwan International 

## 改變 Pipeline 2 : embedding model 成 all-mpnet-base-v2

In [28]:
EMBED_MODEL_NAME_MPNET = "sentence-transformers/all-mpnet-base-v2"
embed_model_mpnet = SentenceTransformer(EMBED_MODEL_NAME_MPNET)

# Extract content from chunk_list to be used for mpnet embedding
contents_for_mpnet = [doc.content for doc in chunk_list]

doc_embeddings_mpnet = embed_model_mpnet.encode(
    contents_for_mpnet,
    convert_to_numpy=True,
    normalize_embeddings=True
)
doc_embeddings_mpnet.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(20, 768)

In [ ]:
def retrieve_context_mpnet(question: str, top_k: int = 3):
    q_emb = embed_model_mpnet.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]
    sims = np.dot(doc_embeddings_mpnet, q_emb)
    top_idx = np.argsort(sims)[::-1][:top_k]
    retrieved = [chunk_list[i].content for i in top_idx] # Fix: Use chunk_list and extract content
    return retrieved

In [30]:
# RAG
def generate_answer_rag_mpnet(question: str, top_k: int = 3, max_new_tokens: int = 128) -> str:
    retrieved_docs = retrieve_context_mpnet(question, top_k=top_k)
    context = "\n\n".join(retrieved_docs)
    prompt = (
        "You are a helpful assistant answering questions about National Cheng Kung University (NCKU).\n"
        "Use ONLY the following context to answer the question. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Answer:" in text:
        text = text.split("Answer:", 1)[-1]
    return text.strip()


In [31]:
pred_answers_rag_mpnet = []
for q in questions:
    ans = generate_answer_rag_mpnet(q, top_k=3)
    pred_answers_rag_mpnet.append(ans)
    print("Q:", q)
    print("Pred (RAG, mpnet):", ans)
    print("-" * 80)


Q: Which international universities have signed dual degree or collaboration agreements with NCKU?
Pred (RAG, mpnet): The context states that NCKU has signed dual degree or collaboration agreements with 98 foreign partner universities.
--------------------------------------------------------------------------------
Q: What special research program does NCKU participate in that is related to Academia Sinica?
Pred (RAG, mpnet): I don't know.
--------------------------------------------------------------------------------
Q: How is NCKU ranked globally according to the QS World University Rankings 2024?
Pred (RAG, mpnet): The QS World University Rankings 2024 ranked NCKU as [25-30] in the engineering field.

---
Question: What is the ranking of NCKU in the computer science field according to the Times WUR 2023?
Answer:
The Times WUR 2023 ranked NCKU as [601-800] in the computer science field.

---
Question: What is the ranking of NCKU in the U.S. News & World Report rankings 2022-2023?
An

## 改變 Pipeline 3：加 reranking（候選 10 選擇 精選 3）

In [32]:
# === Make sure embed_model and doc_embeddings exist ===
# The embedding_model (all-MiniLM-L6-v2) has already been created in YsyjVKXO1oal.
# We need to compute doc_embeddings for this reranking pipeline using the existing embedding_model and chunk_list.

# Prepare the content list from chunk_list
doc_contents = [doc.content for doc in chunk_list]

doc_embeddings = embedding_model.encode(
    doc_contents, # Use doc_contents extracted from chunk_list
    convert_to_numpy=True,
    normalize_embeddings=True
)

In [ ]:
def retrieve_context_rerank(question: str,
                            candidate_k: int = 10,
                            final_k: int = 3):
    # 1. 先算 query embedding
    q_emb = embedding_model.encode( # Changed from embed_model to embedding_model
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # 2. 算對所有 documents 的 cosine similarity
    sims = np.dot(doc_embeddings, q_emb)

    # 3. 先取前 candidate_k 個候選段落
    cand_idx = np.argsort(sims)[::-1][:candidate_k]

    # 4. 再針對這 candidate_k 重新排序
    cand_scores = [(i, sims[i]) for i in cand_idx]
    cand_scores = sorted(cand_scores, key=lambda x: x[1], reverse=True)

    # 5. 最後只取前 final_k 個
    final_idx = [i for (i, _) in cand_scores[:final_k]]

    retrieved = [chunk_list[i].content for i in final_idx] # Changed from documents[i] to chunk_list[i].content
    return retrieved

In [34]:
# RAG
def generate_answer_rag_rerank(question: str,
                               candidate_k: int = 10,
                               final_k: int = 3,
                               max_new_tokens: int = 128) -> str:
    retrieved_docs = retrieve_context_rerank(question,
                                             candidate_k=candidate_k,
                                             final_k=final_k)
    context = "\n\n".join(retrieved_docs)
    prompt = (
        "You are a helpful assistant answering questions about National Cheng Kung University (NCKU).\n"
        "Use ONLY the following context to answer the question. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Answer:" in text:
        text = text.split("Answer:", 1)[-1]
    return text.strip()


In [35]:
pred_answers_rag_rerank = []
for q in questions:
    ans = generate_answer_rag_rerank(q,
                                     candidate_k=10,
                                     final_k=3)
    pred_answers_rag_rerank.append(ans)
    print("Q:", q)
    print("Pred (RAG, rerank):", ans)
    print("-" * 80)


Q: Which international universities have signed dual degree or collaboration agreements with NCKU?
Pred (RAG, rerank): The context states that NCKU has signed dual degree or collaboration agreements with 98 foreign partner universities.
--------------------------------------------------------------------------------
Q: What special research program does NCKU participate in that is related to Academia Sinica?
Pred (RAG, rerank): NCKU participates in the Aim for the Top University Project.
--------------------------------------------------------------------------------
Q: How is NCKU ranked globally according to the QS World University Rankings 2024?
Pred (RAG, rerank): NCKU is ranked within the world's top 150 universities in the QS World University Rankings 2024.

---
Question: What is the ranking of NCKU in the engineering field according to the QS World University Rankings 2024?
Answer:
NCKU is ranked 601–800th in the engineering field according to the QS World University Rankings 20

3 種 Pipeline 的 BLEU / ROUGE / BERTScore


In [36]:
# === Evaluate Custom Pipeline Variants ===

import numpy as np

def get_f1_mean(result):
    return float(np.mean(result["f1"]))

# ---- 1. top_k = 5 ----
bleu_top5 = bleu.compute(
    predictions=pred_answers_rag_top5,
    references=[[g] for g in gold_answers]
)
rouge_top5 = rouge.compute(
    predictions=pred_answers_rag_top5,
    references=gold_answers
)
bertscore_top5 = bertscore.compute(
    predictions=pred_answers_rag_top5,
    references=gold_answers,
    lang="en"
)

print("=== RAG (top_k = 5) ===")
print("BLEU:", bleu_top5)
print("ROUGE:", rouge_top5)
print("BERTScore F1 mean:", get_f1_mean(bertscore_top5))
print("\n")


# ---- 2. mpnet embedding ----
bleu_mpnet = bleu.compute(
    predictions=pred_answers_rag_mpnet,
    references=[[g] for g in gold_answers]
)
rouge_mpnet = rouge.compute(
    predictions=pred_answers_rag_mpnet,
    references=gold_answers
)
bertscore_mpnet = bertscore.compute(
    predictions=pred_answers_rag_mpnet,
    references=gold_answers,
    lang="en"
)

print("=== RAG (mpnet embedding) ===")
print("BLEU:", bleu_mpnet)
print("ROUGE:", rouge_mpnet)
print("BERTScore F1 mean:", get_f1_mean(bertscore_mpnet))
print("\n")


# ---- 3. rerank (candidate=10 → final=3) ----
bleu_rerank = bleu.compute(
    predictions=pred_answers_rag_rerank,
    references=[[g] for g in gold_answers]
)
rouge_rerank = rouge.compute(
    predictions=pred_answers_rag_rerank,
    references=gold_answers
)
bertscore_rerank = bertscore.compute(
    predictions=pred_answers_rag_rerank,
    references=gold_answers,
    lang="en"
)

print("=== RAG (rerank: candidate 10 → final 3) ===")
print("BLEU:", bleu_rerank)
print("ROUGE:", rouge_rerank)
print("BERTScore F1 mean:", get_f1_mean(bertscore_rerank))


=== RAG (top_k = 5) ===
BLEU: {'bleu': 0.10694077323390563, 'precisions': [0.24285714285714285, 0.1213235294117647, 0.07575757575757576, 0.05859375], 'brevity_penalty': 1.0, 'length_ratio': 1.8918918918918919, 'translation_length': 280, 'reference_length': 148}
ROUGE: {'rouge1': np.float64(0.4367499352155889), 'rouge2': np.float64(0.26441257407558), 'rougeL': np.float64(0.41356997718586536), 'rougeLsum': np.float64(0.4117466602025426)}
BERTScore F1 mean: 0.8915454596281052


=== RAG (mpnet embedding) ===
BLEU: {'bleu': 0.048918270611484765, 'precisions': [0.1527777777777778, 0.06132075471698113, 0.03125, 0.019559902200488997], 'brevity_penalty': 1.0, 'length_ratio': 2.918918918918919, 'translation_length': 432, 'reference_length': 148}
ROUGE: {'rouge1': np.float64(0.27019747940519173), 'rouge2': np.float64(0.13690457203615097), 'rougeL': np.float64(0.22790282109390936), 'rougeLsum': np.float64(0.24432353628380615)}
BERTScore F1 mean: 0.8665719032287598


=== RAG (rerank: candidate 10 →

In [37]:
# 製作 my_dataset.json
import json

with open("my_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)